In [ ]:
# ================================================================
# Direct file read from Colab
# Figure texts and outputs translated to English
# ================================================================

# If needed, uncomment:
# !pip install -q pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings

warnings.filterwarnings("ignore")

# ================================================================
# 1. LOAD DATASET
# ================================================================
file_path = "Table1.csv"

def load_csv() -> pd.DataFrame:
    df = pd.read_csv(file_path, low_memory=False)
    print(f"File loaded successfully: {file_path}")
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
    return df


# ================================================================
# 2. FAIL-SAFE DETECTOR (find the real time column)
# ================================================================
def extract_minutes(val):
    if pd.isna(val):
        return np.nan

    s = str(val).lower().strip()

    if s in ["nan", "none", "", "unchecked", "false", "0", "não se aplica", "nao se aplica"]:
        return np.nan

    if ":" in s:
        try:
            p = s.split(":")
            return int(p[0]) * 60 + int(p[1])
        except Exception:
            pass

    nums = re.findall(r"\d+", s)
    if not nums:
        return np.nan

    v = float(nums[0])

    if ("h" in s or "hora" in s) and v <= 5:
        return v * 60

    return v


# ================================================================
# 3. EXTRACT DATA AND QUALITY INDICATOR
# ================================================================
def prepare_data(df: pd.DataFrame):
    candidate_cols = [
        c for c in df.columns
        if any(p in c.lower() for p in ["tempo", "dura", "minutos", "atendimento"])
        and "especialidade" not in c.lower()
    ]

    best_column = None
    max_valid = -1

    for c in candidate_cols:
        valid_count = df[c].apply(extract_minutes).notna().sum()
        if valid_count > max_valid:
            max_valid = valid_count
            best_column = c

    col_time = best_column
    specialty_cols = [c for c in df.columns if "especialidade" in c.lower()]
    ciap_cols = [c for c in df.columns if "ciap" in c.lower()]
    col_ciap = ciap_cols[0] if ciap_cols else None

    if not col_time or max_valid == 0:
        raise ValueError("Critical failure: no time column with real numeric values was found.")

    print("=" * 80)
    print("COLUMN DIAGNOSTIC")
    print(f" -> The detected consultation time column is: '{col_time}'")
    print(f" -> It contains {max_valid} consultations with numeric time values.")
    print("=" * 80)

    df = df.copy()
    df["Consultation_Time_Minutes"] = df[col_time].apply(extract_minutes)

    df_consultations = df[(df["Repeat Instrument"].notna()) | (df["Consultation_Time_Minutes"].notna())].copy()
    if df_consultations.empty:
        df_consultations = df.copy()

    total_consultations = len(df_consultations)
    filled_count = df_consultations["Consultation_Time_Minutes"].notna().sum()
    blank_count = total_consultations - filled_count

    pct_filled = (filled_count / total_consultations) * 100 if total_consultations > 0 else 0
    pct_blank = (blank_count / total_consultations) * 100 if total_consultations > 0 else 0

    df_valid = df_consultations.dropna(subset=["Consultation_Time_Minutes"]).copy()

    return (
        df,
        df_valid,
        col_time,
        specialty_cols,
        col_ciap,
        total_consultations,
        filled_count,
        blank_count,
        pct_filled,
        pct_blank,
    )


# ================================================================
# 4. SPECIALTY CROSSING (real data)
# ================================================================
def extract_specialty_time_crossing(dataframe, specialty_cols, col_ciap):
    records = []
    positive_values = ["checked", "sim", "yes", "1", "true", "verdadeiro"]
    negative_values = ["nan", "none", "", "unchecked", "false", "0", "não se aplica", "nao se aplica", "não", "nao"]

    for _, row in dataframe.iterrows():
        t = row["Consultation_Time_Minutes"]
        ciap = str(row[col_ciap]) if col_ciap else "Unknown"

        for c in specialty_cols:
            v = str(row[c]).strip().lower()
            if pd.isna(row[c]) or v in negative_values:
                continue

            if "choice=" in c.lower():
                if v in positive_values:
                    specialty = c.split("choice=")[-1].replace(")", "").strip()
                    records.append(
                        {
                            "Specialty": specialty,
                            "Consultation_Time_Minutes": t,
                            "CIAP": ciap,
                        }
                    )
            else:
                records.append(
                    {
                        "Specialty": str(row[c]).strip().title(),
                        "Consultation_Time_Minutes": t,
                        "CIAP": ciap,
                    }
                )

    return pd.DataFrame(records).drop_duplicates()


# ================================================================
# 5. OUTLIER MATH (IQR)
# ================================================================
def split_outliers(df_time: pd.DataFrame):
    q1 = df_time["Consultation_Time_Minutes"].quantile(0.25)
    q3 = df_time["Consultation_Time_Minutes"].quantile(0.75)
    iqr = q3 - q1

    lower_limit = max(1, q1 - 1.5 * iqr)
    upper_limit = q3 + 1.5 * iqr

    df_normal = df_time[
        (df_time["Consultation_Time_Minutes"] >= lower_limit)
        & (df_time["Consultation_Time_Minutes"] <= upper_limit)
    ].copy()

    df_outliers = df_time[
        (df_time["Consultation_Time_Minutes"] < lower_limit)
        | (df_time["Consultation_Time_Minutes"] > upper_limit)
    ].copy()

    dirty_mean = df_time["Consultation_Time_Minutes"].mean()
    clean_mean = df_normal["Consultation_Time_Minutes"].mean()

    return df_normal, df_outliers, lower_limit, upper_limit, dirty_mean, clean_mean


# ================================================================
# 6. GENERATE FIGURE (real data without outliers)
# ================================================================
def make_figure(df_normal: pd.DataFrame):
    top_specialties = df_normal["Specialty"].value_counts().nlargest(10).index
    df_plot = df_normal[df_normal["Specialty"].isin(top_specialties)].copy()

    specialty_order = (
        df_plot.groupby("Specialty")["Consultation_Time_Minutes"]
        .median()
        .sort_values(ascending=False)
        .index
    )

    plt.figure(figsize=(14, 8))
    sns.set_theme(style="whitegrid")
    palette = sns.color_palette("plasma_r", len(specialty_order))

    ax = sns.violinplot(
        data=df_plot,
        x="Consultation_Time_Minutes",
        y="Specialty",
        order=specialty_order,
        palette=palette,
        inner=None,
        alpha=0.5,
        linewidth=0.5,
        scale="width",
    )

    sns.stripplot(
        data=df_plot,
        x="Consultation_Time_Minutes",
        y="Specialty",
        order=specialty_order,
        color="black",
        alpha=0.3,
        size=3.5,
        jitter=0.25,
    )

    sns.pointplot(
        data=df_plot,
        x="Consultation_Time_Minutes",
        y="Specialty",
        order=specialty_order,
        estimator=np.mean,
        color="red",
        markers="D",
        linestyles="",
        errwidth=0,
        label="Mean",
    )

    clean_mean = df_plot["Consultation_Time_Minutes"].mean()

    plt.title(
        f"Average Teleconsultation Duration (Filled Records Only | Without Outliers)\n"
        f"Clean Global Mean: {clean_mean:.0f} min",
        fontsize=15,
        fontweight="bold",
        pad=15,
    )
    plt.xlabel("Consultation Time (Minutes)", fontsize=13)
    plt.ylabel("")

    statistics = (
        df_plot.groupby("Specialty")["Consultation_Time_Minutes"]
        .agg(["count", "median"])
        .reindex(specialty_order)
    )

    new_labels = [
        f"{str(spec)[:25]}\n(n={int(row['count'])} | Med. {row['median']:.0f}m)"
        for spec, row in statistics.iterrows()
    ]
    ax.set_yticklabels(new_labels)

    plt.tight_layout()
    plt.show()

    return df_plot, specialty_order


# ================================================================
# 7. EXPORT STATISTICAL TABLE
# ================================================================
def export_table(df_plot: pd.DataFrame, specialty_order):
    df_table = (
        df_plot.groupby("Specialty")["Consultation_Time_Minutes"]
        .agg(
            Consultation_Count="count",
            Minimum_Time="min",
            Mean_Time="mean",
            Median_Time="median",
            Maximum_Time="max",
            Standard_Deviation="std",
        )
        .reset_index()
    )

    time_cols = ["Minimum_Time", "Mean_Time", "Median_Time", "Maximum_Time", "Standard_Deviation"]
    df_table[time_cols] = df_table[time_cols].round(1)

    df_table = df_table.set_index("Specialty").reindex(specialty_order).reset_index()

    output_table_name = "Consultation_Time_by_Specialty_Statistical_Table_EN.csv"
    df_table.to_csv(output_table_name, index=False, encoding="utf-8-sig")

    return df_table, output_table_name


# ================================================================
# 8. EXECUTIVE REPORTS IN OUTPUT
# ================================================================
def print_reports(
    total_consultations,
    filled_count,
    blank_count,
    pct_filled,
    pct_blank,
    lower_limit,
    upper_limit,
    clean_mean,
    dirty_mean,
    df_outliers,
    output_table_name,
):
    print("\n" + "=" * 80)
    print("QUALITY INDICATOR: CONSULTATION TIME RECORD COMPLETENESS")
    print("=" * 80)
    print(f"Total consultations analyzed: {total_consultations}")
    print(f" -> Valid and filled for the chart: {filled_count} ({pct_filled:.1f}%)")
    print(f" -> Left blank (excluded):          {blank_count} ({pct_blank:.1f}%)")

    if pct_blank > 20:
        print("\nWARNING: More than 20% of consultations have no recorded consultation time.")

    print("\n" + "=" * 80)
    print(f"OUTLIER AUDIT (Cases outside the normal range of {lower_limit:.0f}m to {upper_limit:.0f}m)")
    print("=" * 80)
    print(f" - Real Mean (Without Outliers): {clean_mean:.1f} minutes")
    print(f" - Distorted Mean (With Outliers): {dirty_mean:.1f} minutes")
    print(f"Total extreme cases ignored in the chart: {len(df_outliers)} cases\n")

    if not df_outliers.empty:
        high_outliers = df_outliers[df_outliers["Consultation_Time_Minutes"] > upper_limit]
        if not high_outliers.empty:
            print(f"EXTREMELY LONG CONSULTATIONS (> {upper_limit:.0f} min): {len(high_outliers)} cases")
            print("  Top affected specialties:")
            print("  " + "\n  ".join(high_outliers["Specialty"].value_counts().head(3).to_string().split("\n")))
            print("\n  Top CIAPs (diagnoses) that pushed time upward:")
            print("  " + "\n  ".join(high_outliers["CIAP"].value_counts().head(3).to_string().split("\n")))
            print("-" * 50)

    print(f"\nBase chart table saved as: '{output_table_name}'.\n")


# ================================================================
# RUN EVERYTHING
# ================================================================
def main():
    df = load_csv()

    (
        df,
        df_valid,
        col_time,
        specialty_cols,
        col_ciap,
        total_consultations,
        filled_count,
        blank_count,
        pct_filled,
        pct_blank,
    ) = prepare_data(df)

    df_time = extract_specialty_time_crossing(df_valid, specialty_cols, col_ciap)

    if df_time.empty:
        print("Warning: time and specialty were filled in separate forms. Activating data cascade fallback...")
        df_fallback = df.sort_values(by=["Record ID", "Repeat Instance"]).copy()
        df_fallback[specialty_cols] = df_fallback.groupby("Record ID")[specialty_cols].ffill()
        df_valid_fallback = df_fallback.dropna(subset=["Consultation_Time_Minutes"])
        df_time = extract_specialty_time_crossing(df_valid_fallback, specialty_cols, col_ciap)

    if df_time.empty:
        raise ValueError("Fatal error: it was not possible to align consultation time and specialty data.")

    df_normal, df_outliers, lower_limit, upper_limit, dirty_mean, clean_mean = split_outliers(df_time)

    df_plot, specialty_order = make_figure(df_normal)

    _, output_table_name = export_table(df_plot, specialty_order)

    print_reports(
        total_consultations,
        filled_count,
        blank_count,
        pct_filled,
        pct_blank,
        lower_limit,
        upper_limit,
        clean_mean,
        dirty_mean,
        df_outliers,
        output_table_name,
    )


main()